# 🎵 Song Lyrics Analyzer — GPU Analysis Pipeline

Run the full NLP pipeline on Google Colab's T4 GPU (free tier).

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Click the 🔑 icon (left sidebar) and add these secrets:
   - `GROQ_API_KEY`
   - `SUPABASE_URL`
   - `SUPABASE_KEY`

Then run cells top-to-bottom.

In [ ]:
# 1. Clone the repo
!git clone https://github.com/harshitjain25/lyrics-analyzer.git
%cd lyrics-analyzer

In [ ]:
# 2. Install pipeline dependencies (~5-8 min)
!pip install -q -r requirements-pipeline.txt

In [ ]:
# 3. Load API keys from Colab Secrets
import os
from google.colab import userdata

os.environ['GROQ_API_KEY']  = userdata.get('GROQ_API_KEY')
os.environ['SUPABASE_URL']  = userdata.get('SUPABASE_URL')
os.environ['SUPABASE_KEY']  = userdata.get('SUPABASE_KEY')
os.environ['DEMO_MODE']     = 'false'
os.environ['MOCK_MODE']     = 'false'
print('Secrets loaded ✓')

In [ ]:
# 4. Verify GPU is active
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# 5. Check how many songs are in Supabase
from src.database.db_manager import DBManager
db = DBManager()
songs_df = db.get_songs()
print(f'Songs in database: {len(songs_df)}')
songs_df.head()

In [ ]:
# 6. Run the full analysis pipeline (~15-30 min on T4 GPU)
!python scripts/run_analysis.py

In [ ]:
# 7. Save embeddings + model to Google Drive for your dashboard
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/lyrics-analyzer-artifacts
!cp -r data/embeddings/* /content/drive/MyDrive/lyrics-analyzer-artifacts/
!ls -la /content/drive/MyDrive/lyrics-analyzer-artifacts/

In [ ]:
# 8. Optional: download embeddings.npy to your computer directly
from google.colab import files
files.download('data/embeddings/embeddings.npy')
files.download('data/embeddings/song_ids.json')

## Done! ✅

Supabase now has:
- Topic labels
- Sentiment scores per song
- Emotion scores per song
- Aspect-level sentiments

On your Mac, copy `embeddings.npy` and `song_ids.json` into `data/embeddings/` to enable the semantic search page.

Then run `streamlit run dashboard/app.py` with `DEMO_MODE=false` to see live data.